#INGEST FACT DATA INTO BRONZE

In [0]:
from pyspark.sql.types import *
import pyspark.sql.functions as F

dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
dbutils.widgets.text("storage_account_name", "storageaccount", "Storage Account Name")
dbutils.widgets.text("container_name", "ecomm-raw-data", "Container Name")

catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

In [0]:
adls_path = (
    f"abfss://{container_name}@"
    f"{storage_account_name}.dfs.core.windows.net/"
    f"order_items/landing/"
)

bronze_checkpoint_path = (
    f"abfss://{container_name}@"
    f"{storage_account_name}.dfs.core.windows.net/"
    f"checkpoint/bronze/fact_order_items/"
)

In [0]:
(
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", bronze_checkpoint_path)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("rescuedDataColumn", "_rescued_data")
    .option("cloudFiles.includeExistingFiles", "true")
    .option("pathGlobFilter", "*.csv")
    .load(adls_path)

    .withColumn("ingest_timestamp", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))

    .writeStream
    .outputMode("append")
    .option("checkpointLocation", bronze_checkpoint_path)
    .trigger(availableNow=True)
    .toTable(f"{catalog_name}.bronze.brz_order_items")
    .awaitTermination()
)

In [0]:
# display(
#     spark.sql(
#         f"SELECT MIN(dt), MAX(dt) FROM {catalog_name}.bronze.brz_order_items"
#     )
# )

# display(
#     spark.sql(
#         f"""
#         SELECT COUNT(*)
#         FROM CLOUD_FILES_STATE(
#         '{bronze_checkpoint_path}'
#         )
#         """
#     )
# )

MIN(dt),MAX(dt)
2024-01-01,2025-08-31


COUNT(*)
609
